In [0]:
%pip install pandas-gbq

In [0]:
# Verificando la versión de Spark
print(f"Spark Version: {spark.version}")


In [0]:
# 1. Definir ruta del dataset
csv_path = "/Volumes/workspace/default/rawdata/market_pipe_thickness_loss_dataset.csv"

# 2. Leer el dataset con Spark
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(csv_path)

# 3. Mostrar los primeros resultados del dataset
print("Dataset de Tuberías cargado exitosamente:")
display(df.limit(10))

In [0]:
# Datos estadísticos
print("Resumen de variables críticas para integridad de tuberías:")
df.select("Thickness_mm", "Max_Pressure_psi", "Temperature_C", "Corrosion_Impact_Percent").summary().show()

# Verificación de datos nulos (Calidad de Datos)
from pyspark.sql.functions import col, count, when
print("Reporte de Calidad de Datos (Nulos):")
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

In [0]:
from pyspark.sql import functions as F

# Renombrando columnas para estandarizar el dataset
df_clean = df.withColumnRenamed("Thickness_mm", "thickness_mm") \
             .withColumnRenamed("Max_Pressure_psi", "max_pressure_psi") \
             .withColumnRenamed("Temperature_C", "temperature_c") \
             .withColumnRenamed("Corrosion_Impact_Percent", "corrosion_impact_pct") \
             .withColumnRenamed("Thickness_Loss_mm", "thickness_loss_mm")

# Crear una columna calculada de ejemplo: Porcentaje de vida útil restante
df_final = df_clean.withColumn("remaining_life_pct", 100 - F.col("corrosion_impact_pct"))

print("Columnas actualizadas exitosamente:")
df_final.select("thickness_mm", "max_pressure_psi", "remaining_life_pct").show(5)

In [0]:
# Definir la ruta del archivo
file_path = "/Volumes/workspace/default/rawdata/market_pipe_thickness_loss_dataset.csv"

# Cargar los datos
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

# Mostrar los datos para validar la ingesta
display(df)

# Guardar como tabla Delta (Para mayor velocidad en el futuro)
df.write.mode("overwrite").saveAsTable("workspace.default.pipeline_raw")
print("¡Datos cargados y convertidos a tabla Delta exitosamente!")

In [0]:
from pyspark.sql import functions as F

# Renombrando columnas para estandarizar el dataset
df_clean = df.withColumnRenamed("Thickness_mm", "thickness_mm") \
             .withColumnRenamed("Max_Pressure_psi", "max_pressure_psi") \
             .withColumnRenamed("Temperature_C", "temperature_c") \
             .withColumnRenamed("Corrosion_Impact_Percent", "corrosion_impact_pct") \
             .withColumnRenamed("Thickness_Loss_mm", "thickness_loss_mm")

# Columnas adicionales para calcular el porcentaje de vida útil restante
df_final = df_clean.withColumn("remaining_life_pct", 100 - F.col("corrosion_impact_pct"))

print("Columnas actualizadas exitosamente:")
df_final.select("thickness_mm", "max_pressure_psi", "remaining_life_pct").show(5)

In [0]:
import pandas as pd
from pyspark.sql import functions as F
import time

# 1. CARGA DE DATOS (PySpark)
path = "/Volumes/workspace/default/rawdata/market_pipe_thickness_loss_dataset.csv"
df_spark = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(path)

# --- ESCENARIO A: ANALÍTICA A GRAN ESCALA (PySpark) ---
# Imagina que procesas 1 millón de sensores en tiempo real.
start_spark = time.time()
res_spark = df_spark.groupBy("Material").agg(
    F.avg("Thickness_mm").alias("avg_thickness"),
    F.max("Max_Pressure_psi").alias("max_pressure")
)
res_spark.show()
end_spark = time.time()
print(f"Tiempo PySpark: {end_spark - start_spark:.4f} segundos")

# --- ESCENARIO B: ANÁLISIS DESCRIPTIVO DETALLADO (Pandas) ---
start_pandas = time.time()
df_pandas = df_spark.pandas_api() # Convertimos a Pandas
res_pandas = df_pandas.groupby("Material").agg({
    "Thickness_mm": "mean",
    "Max_Pressure_psi": "max"
})
print(res_pandas)
end_pandas = time.time()
print(f"Tiempo Pandas: {end_pandas - start_pandas:.4f} segundos")

In [0]:
import pandas_gbq
from google.oauth2 import service_account

# 1. Llevar los datos a Pandas con un límite de seguridad
pdf_final = df_final.limit(50000).toPandas() 

# 2. Gestión de Credenciales
service_account_path = "/Workspace/Users/erlintoneligon@hotmail.com/nexus-pipeline-integrity-analytics/service-account-key.json"
try:
    credentials = service_account.Credentials.from_service_account_file(service_account_path)
    
    # 3. Carga optimizada
    pandas_gbq.to_gbq(
        pdf_final,
        destination_table='nexus_analytics.pipeline_results',
        project_id='solid-setup-428513-c5',
        if_exists='replace',
        credentials=credentials,
        api_method='load_csv', # O 'load_parquet' si el dataset crece
        progress_bar=True
    )
    print(f"✅ ¡Éxito! {len(pdf_final)} registros en BigQuery.")

except Exception as e:
    print(f"❌ Error en la carga: {e}")

In [0]:
from pyspark.sql import functions as F

# 1. Estandarización de nombres (Snake Case para consistencia)
df_std = df_spark.select(
    F.col("Pipe_Size_mm").alias("pipe_size_mm"),
    F.col("Thickness_mm").alias("thickness_mm"),
    F.col("Max_Pressure_psi").alias("max_pressure_psi"),
    F.col("Material").alias("material"),
    F.col("Corrosion_Impact_Percent").alias("corrosion_pct")
)

# 2. Detección de Outliers (Valores atípicos)
# Si la presión es > 2000 psi o la corrosión es > 15%, lo marcamos como 'Crítico'
df_status = df_std.withColumn(
    "integrity_status",
    F.when((F.col("max_pressure_psi") > 2000) | (F.col("corrosion_pct") > 15), "Critical")
    .when(F.col("corrosion_pct") > 10, "Warning")
    .otherwise("Stable")
)

print("Reporte de Integridad por Material:")
df_status.groupBy("material", "integrity_status").count().orderBy("material").show()

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Resumen pequeño desde Spark
pdf_report = df_status.groupBy("material", "integrity_status").count().toPandas()

plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")

# 1. Definimos un diccionario de colores base
# 2. Usamos .get() o una paleta por defecto para evitar el ValueError
base_colors = {
    'Operational': 'green', 
    'Critical': 'red', 
    'Warning': 'orange',
    'Stable': 'skyblue'  # Agregamos la que faltaba
}

# Creamos la paleta para que todos los estados tengan un color 
present_statuses = pdf_report['integrity_status'].unique()
current_palette = {status: base_colors.get(status, 'gray') for status in present_statuses}

# 3. Graficamos
sns.barplot(
    data=pdf_report, 
    x='material', 
    y='count', 
    hue='integrity_status',
    palette=current_palette
)

plt.title('Análisis de Integridad Mecánica - Nexus Data Analytics', fontsize=15)
plt.xlabel('Material de la Tubería', fontsize=12)
plt.ylabel('Cantidad de Segmentos', fontsize=12)
plt.legend(title='Estado de Integridad', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [0]:
from pyspark.sql import functions as F

def analyze_pipeline_integrity(dataframe):
    """
    Función profesional para estandarizar y clasificar la integridad de tuberías.
    Ideal para implementaciones de Oil & Gas.
    """
    # Estandarización
    df_clean = dataframe.select(
        F.col("Pipe_Size_mm").alias("size_mm"),
        F.col("Thickness_mm").alias("thickness_mm"),
        F.col("Max_Pressure_psi").alias("pressure_psi"),
        F.col("Material").alias("material"),
        F.col("Corrosion_Impact_Percent").alias("corrosion_pct")
    )
    
    # Lógica de Negocio: Clasificación de Riesgo
    df_analyzed = df_clean.withColumn(
        "risk_level",
        F.when((F.col("pressure_psi") > 2000) | (F.col("corrosion_pct") > 15), "3-Critical")
        .when(F.col("corrosion_pct") > 10, "2-Warning")
        .otherwise("1-Stable")
    )
    
    return df_analyzed

# Ejecución de la función maestra
nexus_results = analyze_pipeline_integrity(df_spark)
display(nexus_results.limit(10))

Databricks visualization. Run in Databricks to view.

In [0]:
# Exportar df_final a CSV para migración a PostgreSQL
output_path = "/Volumes/workspace/default/rawdata/pipeline_final_export.csv"

# Convertir a Pandas y exportar (para datasets manejables)
# Si el dataset es muy grande, se recomienda usar .write.format("csv").save(...)
df_export = df_final.toPandas()
df_export.to_csv(output_path, index=False)

print(f"✅ Dataset exportado exitosamente:")
print(f"   Ruta: {output_path}")
print(f"   Registros: {len(df_export):,}")
print(f"   Columnas: {list(df_export.columns)}")